In [ ]:
!pip install -U scikit-learn imbalanced-learn
from imblearn.over_sampling import SMOTE
from collections import Counter

In [ ]:
import os
import numpy as np
import pandas as pd
import tensorflow as tf

from tensorflow import keras
from tensorflow.keras import layers
from tensorflow.keras import regularizers
from keras_tuner import HyperParameters
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.metrics import classification_report, confusion_matrix, ConfusionMatrixDisplay, precision_score

In [ ]:
path = '/kaggle/input/zenodo2/Zenodo2'
patient_df = pd.read_csv(os.path.join(path, 'Patientwise_Data.csv'))
image_df = pd.read_csv(os.path.join(path, 'Imagewise_Data.csv'))

#Data Modifying
patient_df.columns = patient_df.columns.str.strip()
image_df.columns = image_df.columns.str.strip()

image_df['Patient ID'] = image_df['Image Name'].str.split('-').str[:2].str.join('-')
category_severity = {'OCA': 3, 'OPMD': 2, 'Benign': 1, 'Healthy': 0}
image_df['severity'] = image_df['Category'].map(category_severity)
patient_categories = image_df.sort_values('severity', ascending=False).drop_duplicates('Patient ID')
final_df = pd.merge(patient_df, patient_categories[['Patient ID', 'Category']], on='Patient ID', how='inner')

print(final_df)

In [4]:
X = final_df.drop(['Patient ID', 'Category', 'Image Count'], axis=1)
Y = final_df['Category']

encoder = LabelEncoder()
y = encoder.fit_transform(Y)

X = pd.get_dummies(X, columns=['Gender', 'Alcohol', 'Smoking', 'Chewing_Betel_Quid'], drop_first=True)

X_train, X_temp, y_train, y_temp = train_test_split(
    X, y,test_size=0.3, random_state=42 ,stratify=y
)
X_test, X_val, y_test, y_val = train_test_split(
    X_temp, y_temp,test_size=0.5, random_state=42 ,stratify=y_temp
)

scaler = StandardScaler()
X_train['Age'] = scaler.fit_transform(X_train[['Age']])
X_test['Age'] = scaler.transform(X_test[['Age']])
X_val['Age'] = scaler.transform(X_val[['Age']])

In [ ]:
X_train_full = pd.concat([X_train, X_val])
y_train_full = np.concatenate([y_train, y_val])

print(f"Original full training set distribution: {Counter(y_train_full)}")
smote = SMOTE(random_state=42)
X_train_resampled, y_train_resampled = smote.fit_resample(X_train_full, y_train_full)
print(f"Resampled training set distribution: {Counter(y_train_resampled)}")

In [21]:
alpha = 0.0015
beta = 0.2
model = keras.Sequential([
    layers.Input(shape=(X_train_resampled.shape[1],)),
    layers.Dense(512, activation='relu', kernel_regularizer=regularizers.l2(alpha)),
    layers.Dense(256, activation='relu', kernel_regularizer=regularizers.l2(alpha)),
    layers.Dropout(beta),
    layers.Dense(64, activation='relu', kernel_regularizer=regularizers.l2(alpha)),
    layers.Dense(32, activation='relu', kernel_regularizer=regularizers.l2(alpha)),
    layers.Dense(16, activation='relu', kernel_regularizer=regularizers.l2(alpha)),
    layers.Dense(4, activation='softmax'),
])


In [ ]:
model.compile(
    optimizer='adam',
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
    )

model.summary()

In [ ]:
train = model.fit(
    X_train_resampled,
    y_train_resampled,
    epochs=200,
    batch_size=32,
    validation_split=0.1,
    verbose=1
)

In [ ]:
y_pred_prob = model.predict(X_test)
y_pred = np.argmax(y_pred_prob, axis=1)

cm = confusion_matrix(y_test, y_pred)
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=encoder.classes_)
disp.plot(cmap="Blues")

In [25]:
print("--- Classification Report ---")
print(classification_report(y_test, y_pred, target_names=encoder.classes_))
precision = precision_score(y_test, y_pred, average='weighted')

from sklearn.metrics import precision_score, recall_score, f1_score, roc_auc_score, roc_curve, auc

precision = precision_score(y_test, y_pred, average='weighted')
recall = recall_score(y_test, y_pred, average='weighted')
f1 = f1_score(y_test, y_pred, average='weighted')
print(f"Precision: {precision}, Recall: {recall}, F1: {f1}")

--- Classification Report ---
              precision    recall  f1-score   support

      Benign       0.50      0.72      0.59        36
     Healthy       0.08      0.14      0.10        14
         OCA       0.33      0.44      0.38         9
        OPMD       0.72      0.27      0.39        48

    accuracy                           0.42       107
   macro avg       0.41      0.40      0.37       107
weighted avg       0.53      0.42      0.42       107

Precision: 0.5307165109034268, Recall: 0.4205607476635514, F1: 0.4209933368811873
